# Setting

In [ ]:
# --- Imports ---
import os
import re
import json
import uuid
import asyncio
from datetime import datetime, timezone
from typing import Dict, List, Any

# --- Dependencies ---
# pip install python-dotenv google-generativeai falkordb
from google import genai
from google.genai import types
from dotenv import load_dotenv
from redis.asyncio import Redis
from redis import Redis as RedisSync

In [ ]:
from pathlib import Path
import sys
PROJECT_ROOT = Path("/Users/thubpham/knowledge_graph_ingestion")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from knowledge_graph_extractor import prompts as PROMPT_TEMPLATES

# Gemini Client

In [ ]:
class GeminiLLMClient:

    """A client to interact with the Google Gemini API."""
    
    def __init__(self, *args, **kwargs):
        api_key = os.getenv('GEMINI_API_KEY')
        self.client = genai.Client(api_key=api_key)
    
    # Safeguard method to extract JSON from LLM output, handling common formatting issues
    def _salvage_json(self, raw_output: str | None) -> Dict[str, Any] | None:
        """Robustly extracts a JSON object from a raw LLM output string."""
        if not raw_output: return None
        
        raw_output = re.sub(r"^```json|^```|```$", "", raw_output.strip(), flags=re.MULTILINE).strip()
        
        try:
            return json.loads(raw_output)
        except json.JSONDecodeError:
            print("Warning: Initial JSON parse failed, attempting to find object within string.")
            match = re.search(r"\{.*\}", raw_output, re.DOTALL)
            if match:
                try:
                    return json.loads(match.group(0))
                except json.JSONDecodeError:
                    print("Error: Could not parse JSON even after salvaging.")
                    return None
        return None
    
    async def generate_json(self, system_prompt: str, user_prompt: str) -> Dict | None:
        """Makes an async call to the Gemini API and returns a parsed JSON object."""
        try:
            response = await self.client.aio.models.generate_content(
                model='gemini-2.5-flash',
                config=types.GenerateContentConfig(
                    temperature=0.1,
                    top_p=0.9,
                    max_output_tokens=55000,
                    system_instruction=system_prompt,
                    thinking_config=types.ThinkingConfig(
                    thinking_budget=1024),
                ),
                contents=user_prompt)

            raw_json = self._salvage_json(response.text)
            if not raw_json:
                print(f"Error: Failed to salvage JSON from LLM response: {response.text}")
            return raw_json
        except Exception as e:
            print(f"An error occurred during Gemini API call: {e}")
            return None

In [35]:
# --- Example: Starting an instance of GeminiLLMClient ---
def gemini(sys_prompt: str, user_prompt: str):
    
    load_dotenv()  # Load environment variables from .env file
    gemini_client = GeminiLLMClient()
    
    system_prompt = sys_prompt
    user_prompt = user_prompt

    response = gemini_client.client.models.generate_content(
        model='gemini-2.5-flash',
        config=types.GenerateContentConfig(
            temperature=0.1,
            top_p=0.9,
            max_output_tokens=55000,
            response_mime_type="application/json",
            system_instruction=system_prompt,
            thinking_config=types.ThinkingConfig(
            thinking_budget=1024),),
        contents=user_prompt
    )
    
    if response:
        print("Agent's response:", response.text)
    else:
        print("Failed to get a valid JSON response.")
    
    return gemini_client._salvage_json(response.text)

# FalkorDB Client

In [ ]:
class FalkorDBClient:
    """A client to interact with a FalkorDB graph database."""
    def __init__(self, host, port, db_name):
        self.db_client = RedisSync(host=host, port=port, decode_responses=True)
        self.graph = self.db_client.graph(db_name)
        print(f"FalkorDBClient connected to graph '{db_name}' at {host}:{port}.")
    
    def _execute_query(self, query, params=None):
        return self.graph.query(query, params)

    async def find_potential_duplicate_nodes(self, entity_names: List[str]) -> List[Dict]:
        """Finds existing nodes that might be duplicates based on name matching."""
        if not entity_names:
            return []
        query = """
        UNWIND $names AS name
        MATCH (n:Node)
        WHERE toLower(n.entity_name) CONTAINS toLower(name)
        RETURN n
        """
        result = self._execute_query(query, {"names": entity_names})
        nodes = [record[0].properties for record in result.result_set]
        print(f"DB: Found {len(nodes)} potential duplicate nodes.")
        return nodes

    async def find_potential_related_edges(self, node_uuids: List[str]) -> List[Dict]:
        """Finds all existing edges connected to a list of potentially duplicate nodes."""
        if not node_uuids:
            return []
        query = """
        UNWIND $uuids AS uuid
        MATCH (n {uuid: uuid})-[r]-()
        RETURN r
        """
        result = self._execute_query(query, {"uuids": node_uuids})
        edges = [record[0].properties for record in result.result_set]
        print(f"DB: Found {len(edges)} potentially related edges.")
        return edges

    async def integrate_graph_data(self, resolution_plan: Dict, candidate_subgraph: Dict):
        """Processes the full resolution plan to create/update nodes and edges."""
        # 1. Handle nodes and create a mapping from temp IDs to permanent UUIDs
        temp_id_to_uuid_map = {}
        candidate_nodes_map = {node['temp_id']: node for node in candidate_subgraph['nodes']}

        for resolution in resolution_plan.get('node_resolutions', []):
            temp_id = resolution['temp_id']
            res_uuid = resolution['resolution']

            if res_uuid == 'NEW':
                new_uuid = str(uuid.uuid4())
                node_data = candidate_nodes_map[temp_id]
                params = {**node_data, "uuid": new_uuid, "uuid_1": new_uuid, "name": node_data['entity_name']}
                self._execute_query(
                    # "CREATE (n:Node $params)", {"params": params}
                    "CREATE (n:Node) SET n = $params", {"params": params} # NEW syntax: CREATE => CREATE and SET
                )
                temp_id_to_uuid_map[temp_id] = new_uuid
                print(f"DB: UPLOADED new node '{node_data['entity_name']}'.")
            else:
                temp_id_to_uuid_map[temp_id] = res_uuid

        # 2. Handle edges using the new mapping
        for resolution in resolution_plan.get('edge_resolutions', []):
            if resolution['resolution'] == 'NEW':
                edge_data = resolution['candidate_edge']
                source_uuid = temp_id_to_uuid_map[edge_data['source_temp_id']]
                target_uuid = temp_id_to_uuid_map[edge_data['target_temp_id']]
                print(f"Edge Data: {edge_data}")
                
                params = {
                    "uuid": str(uuid.uuid4()),
                    "fact_text": edge_data.get('fact_text', ''),
                    "valid_at": edge_data.get('valid_at'),
                    "is_active": True
                }

                # Remove None values from params to avoid setting them in the DB
                params = {k: v for k, v in params.items() if v is not None}

                # Try catch block to handle missing relation_type
                try:
                    relation_type = edge_data['relation_type']
                except KeyError:
                    if 'relation_text' in edge_data:
                        relation_type = edge_data['relation_text']
                        print(f"Using fallback 'relation_text' as relation_type: {relation_type}")
                    else:
                        print(f"Edge missing both 'relation_type' and 'relation_text'. Skipping...")
                        continue  # Skip this edge if neither field is available
                
                query = f"""
                MATCH (a:Node {{uuid: $source_uuid}}), (b:Node {{uuid: $target_uuid}})
                CREATE (a)-[r:`{relation_type}`]->(b)
                SET r = $params
                """
                self._execute_query(query, {
                    "source_uuid": source_uuid, 
                    "target_uuid": target_uuid, 
                    "params": params
                })
                print(f"DB: UPLOADED new edge '{relation_type}'.")

        # 3. Handle invalidations
        for inv in resolution_plan.get('invalidated_edges', []):
            edge_uuid = inv['uuid']
            self._execute_query(
                "MATCH ()-[r {uuid: $uuid}]->() SET r.is_active = false", 
                {"uuid": edge_uuid}
            )
            print(f"DB: INVALIDATED edge {edge_uuid[:8]}... due to: {inv['reason']}")

In [ ]:
class KnowledgeGraphProcessor:
    """Orchestrates the entire KG building process for a text chunk."""
    def __init__(self, db_client: FalkorDBClient, llm_client: GeminiLLMClient):
        self.db = db_client
        self.llm = llm_client
        print("KnowledgeGraphProcessor initialized.")

    async def add_episode(self, text_chunk: str):
        """Processes a new chunk of text and updates the knowledge graph."""
        print(f"\n===== Processing new episode... =====")

        # 1. Call 1: One-Shot Extraction
        print("\n--- Step 1: Kicking off Extraction Call ---")
        extraction_response = await self.llm.generate_json(
            PROMPT_TEMPLATES.extraction_system_prompt,
            PROMPT_TEMPLATES.extraction_user_prompt.format(
                reference_time=datetime.now(timezone.utc).isoformat() + "Z",
                text_chunk=text_chunk
            )
        )
        if not extraction_response or "candidate_subgraph" not in extraction_response:
            print("Extraction failed or returned empty. Concluding process.")
            return
        
        candidate_subgraph = extraction_response["candidate_subgraph"]
        if not candidate_subgraph.get('nodes'):
            print("Extraction yielded no nodes. Concluding process.")
            return
        print("LLM Response (Extraction) received and parsed.")

        # 2. Query DB for potential duplicates
        node_names = [node['entity_name'] for node in candidate_subgraph['nodes']]
        potential_nodes = await self.db.find_potential_duplicate_nodes(node_names)
        potential_node_uuids = [node['uuid'] for node in potential_nodes]
        potential_edges = await self.db.find_potential_related_edges(potential_node_uuids)

        # 3. Call 2: Holistic Resolution
        print("\n--- Step 2: Kicking off Resolution Call ---")
        resolution_plan = await self.llm.generate_json(
            PROMPT_TEMPLATES.holistic_resolution_system,
            PROMPT_TEMPLATES.holistic_resolution_user.format(
                candidate_subgraph=json.dumps(candidate_subgraph, indent=2),
                existing_nodes=json.dumps(potential_nodes, indent=2),
                existing_edges=json.dumps(potential_edges, indent=2)
            )
        )
        if not resolution_plan:
            print("Resolution call failed. Aborting integration.")
            return
        print("LLM Response (Resolution) received and parsed.")

        # 4. Integrate results into DB
        print("\n--- Step 3: Integrating results into DB ---")
        await self.db.integrate_graph_data(resolution_plan, candidate_subgraph)
        
        print("===== Episode processing complete. =====")

# Graph Update

In [ ]:
async def populate_main():
    """Main execution function."""
    load_dotenv()
    api_key = os.getenv('GEMINI_API_KEY')
    if not api_key:
        raise ValueError("GEMINI_API_KEY not found in .env file.")

    # --- Configuration ---
    FALKORDB_CONFIG = {
        "host": 'localhost',
        "port": 6379,
        "db_name": 'usr-sim-march-6' # Use a descriptive name for your graph
    }
    GEMINI_CONFIG = {
        "api_key": api_key,
        "model_name": "gemini-2.5-flash"
    }

    # --- Initialization ---
    db_client = FalkorDBClient(**FALKORDB_CONFIG)
    llm_client = GeminiLLMClient(**GEMINI_CONFIG)
    processor = KnowledgeGraphProcessor(db_client=db_client, llm_client=llm_client)

    # try: 
    #     await processor.add_episode(usr_query)
    # except Exception as e:
    #     print(f"An error occurred during processing: {e}")

    await processor.add_episode(usr_query)
    
    print(f"--- Finish parsing ---")
    print("Waiting 1 second to respect rate limits...\n")

    await asyncio.sleep(1)

In [ ]:
await populate_main()

In [ ]:
# Checking Graph Content
from redis import Redis

client = Redis(host='localhost', port=6379, decode_responses=True)

# Print all graph names to verify connection
result = client.execute_command("GRAPH.LIST")
print(f"Available graphs in FalkorDB: {result}")

# Check if our graph is in the list
if 'usr-sim-march-6' in result:
    graph = client.graph('usr-sim-march-6')
    try: 
        result = graph.query("MATCH (n:Node) RETURN count(n)")
        print(f"Node in graph usr-sim-march-6: {result.result_set}")
    except Exception as e:
        print(f"Error querying graph: {e}")

# Graph Query

In [31]:
usr_query = "Does the chicken exist first or the egg exist first?"
sys_prompt = """You are a helpful assistant that provides information in JSON format. You will be given a user query and your task is to extract its key entities. You will return a JSON object with a list of entities, where each entity is a string. 

Example query: Does the chicken exist first or the egg exist first?
Return format:
{{
    "entities": [
        "chicken",
        "egg"
    ]
}}
"""

In [39]:
async def extract_entities_from_queries(query: str) -> List[str] | None:
    """
    Searches the graph for entities related to the user query and formats the results
    into a string context for the RAG model.
    """
    print(f"\n--- Searching Graph for query: '{query}' ---")

    gemini_returned_entities = gemini(sys_prompt, usr_query)

    if not gemini_returned_entities:
        print("No response from Gemini. Cannot extract entities.")
        return None
    
    entities_in_query = gemini_returned_entities['entities']
    
    if not entities_in_query:
        print("No entities identified in the query.")
        return None

    print(f"Identified entities: {entities_in_query}")
    return entities_in_query

In [40]:
async def retrieve_node(entity_list: List[str] | None, db_client: FalkorDBClient) -> List[Dict] | None:
    """Find nodes matching the identified entities"""

    if not entity_list:
        print("No entities provided for node retrieval.")
        return None
    
    query = """
    UNWIND $names AS entityName
    MATCH (n:Node)
    WHERE toLower(n.entity_name) CONTAINS toLower(entityName)
    RETURN n.uuid AS uuid, n.entity_name AS name, n.summary AS summary
    """
    
    results = db_client._execute_query(query, {"names": entity_list})
    
    found_nodes = results.result_set
    if not found_nodes:
        print("No matching nodes found in the database.")
        return None
    
    nodes = [
        {
            "uuid": record[0],
            "name": record[1],
            "summary": record[2]
        }
        for record in found_nodes
    ]

    return nodes


In [41]:
async def format_node_as_context(nodes: List[Dict] | None) -> List[str] | None:
    if not nodes:
        print("No node data provided for formatting.")
        return None 
    context = ["Found the followuing nodes in the graph that might be relevant to user's query:"]
    for node in nodes:
        context.append(f"\nNode Name: {node['name']}\nSummary: {node['summary']}")
    return context

In [25]:
async def retrieve_edges(node_uuids: List[str], db_client: FalkorDBClient) -> List[Dict] | None:

    if not node_uuids:
        print("No node UUIDs provided for edge retrieval.")
        return None
    
    edge_query = """
    UNWIND $uuids AS node_uuid
    MATCH (a:Node {uuid: node_uuid})-[r]->(b:Node)
    WHERE r.is_active = true
    RETURN a.entity_name, type(r), b.entity_name, r.fact_text
    """

    edge_results = db_client._execute_query(edge_query, {"uuids": node_uuids})

    edges = [
        {
            "source": record[0],
            "relation": record[1],
            "target": record[2],
            "fact_text": record[3]
        }
        for record in edge_results.result_set
    ]

    return edges

In [42]:
async def format_edges_as_context(edges: List[Dict] | None) -> List[str] | None:
    if not edges:
        print("No edge data provided for formatting.")
        return None 
    context = ["Found the following edges in the graph that might be relevant to user's query:"]
    for edge in edges:
        context.append(f"\nFact: {edge['fact_text']}\nConnect ({edge['source']} --[{edge['relation']}]-> {edge['target']})")
    return context

In [43]:
async def graph_query_main(usr_query: str):

    load_dotenv()
    FALKORDB_CONFIG = {
        "host": 'localhost',
        "port": 6379,
        "db_name": 'usr-sim-march-6' # Use a descriptive name for your graph
    }
    db_client = FalkorDBClient(**FALKORDB_CONFIG)  

    entities = await extract_entities_from_queries(usr_query)
    nodes = await retrieve_node(entities, db_client)
    node_context = await format_node_as_context(nodes)
    
    if not nodes:
        print("No relevant nodes found. Ending query process.")
        return
    node_uuids = [node['uuid'] for node in nodes]
    edges = await retrieve_edges(node_uuids, db_client)
    edge_context = await format_edges_as_context(edges)

    context = ""
    if node_context:
        context += "\n".join(node_context) + "\n"
    if edge_context:
        context += "\n".join(edge_context)
    
    return context


In [44]:
await graph_query_main(usr_query)

FalkorDBClient connected to graph 'usr-sim-march-6' at localhost:6379.

--- Searching Graph for query: 'Does the chicken exist first or the egg exist first?' ---
Agent's response: {
    "entities": [
        "chicken",
        "egg"
    ]
}
Identified entities: ['chicken', 'egg']
No matching nodes found in the database.
No node data provided for formatting.
No relevant nodes found. Ending query process.


# Intention Routing